In [3]:
from pathlib import Path
import re

ROUND_FILE_RE = re.compile(r"(?i)^(?:r|round)\s*(\d+)\.txt$")
LINE_RE = re.compile(r"^\s*(\d*)\s+(.+?)\s{2,}(.+?)\s{2,}(\d+(?:-\d+)?)\s*$")

def find_round_files(folder: str | Path):
    folder = Path(folder)
    files = []
    for p in folder.iterdir():
        if p.is_file() and p.suffix.lower() == ".txt":
            m = ROUND_FILE_RE.match(p.name)
            if m:
                files.append((int(m.group(1)), p))
    return [p for _, p in sorted(files, key=lambda x: x[0])]

def extract_names_from_rounds(folder: str | Path) -> set[str]:
    names = set()
    for path in find_round_files(folder):
        for line in path.read_text(encoding="utf-8").splitlines():
            if not line.strip() or "Table" in line or set(line.strip()) <= {"-"}:
                continue
            if line.strip().lower().startswith("round"):
                continue

            m = LINE_RE.match(line)
            if not m:
                continue

            _, player, opponent, _ = m.groups()
            player = player.strip()
            opponent = opponent.strip()

            for n in (player, opponent):
                if n == "***Bye***":
                    continue
                if n.upper().startswith("[REDACTED]"):
                    continue
                names.add(n)
    return names

def update_player_list_txt(player_list_path: str | Path, new_names: set[str]) -> list[str]:
    player_list_path = Path(player_list_path)
    existing = set()

    if player_list_path.exists():
        existing = {ln.strip() for ln in player_list_path.read_text(encoding="utf-8").splitlines() if ln.strip()}

    to_add = sorted(new_names - existing)

    if to_add:
        with player_list_path.open("a", encoding="utf-8") as f:
            # ensure file ends with newline
            if player_list_path.stat().st_size > 0:
                f.write("\n")
            f.write("\n".join(to_add) + "\n")

    return to_add

# ---- USAGE ----
rounds_folder = r"2_20_26"
player_list_file = r"Player List.txt"

names_seen = extract_names_from_rounds(rounds_folder)
added = update_player_list_txt(player_list_file, names_seen)

print(f"Found {len(names_seen)} unique non-bye names in rounds.")
print(f"Appended {len(added)} new names to {player_list_file}:")
for n in added:
    print(" -", n)

Found 18 unique non-bye names in rounds.
Appended 3 new names to Player List.txt:
 - Ash DerManouelian
 - Evan Hamlin
 - Jason Weiss
